# Task 12: Unsupervised Learning & Dimensionality Reduction

## Introduction to Unsupervised Learning

Unsupervised Learning involves finding patterns in data without labeled responses. Key techniques include clustering, dimensionality reduction, and association rule mining.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 2. K-Means Clustering

### Implementing K-Means from Scratch

In [ ]:
class KMeansFromScratch:
    """K-Means Clustering implementation from scratch"""
    
    def __init__(self, n_clusters=3, max_iter=100, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.random_state = random_state
        self.centroids = None
        self.labels = None
    
    def fit(self, X):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        
        # Initialize centroids randomly
        idx = np.random.choice(n_samples, self.n_clusters, replace=False)
        self.centroids = X[idx]
        
        for _ in range(self.max_iter):
            # Assign points to nearest centroid
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)
            
            # Update centroids
            new_centroids = np.zeros_like(self.centroids)
            for k in range(self.n_clusters):
                if np.sum(self.labels == k) > 0:
                    new_centroids[k] = X[self.labels == k].mean(axis=0)
                else:
                    new_centroids[k] = self.centroids[k]
            
            # Check convergence
            if np.allclose(self.centroids, new_centroids):
                break
            
            self.centroids = new_centroids
        
        return self
    
    def _compute_distances(self, X):
        distances = np.zeros((X.shape[0], self.n_clusters))
        for k in range(self.n_clusters):
            distances[:, k] = np.sqrt(np.sum((X - self.centroids[k])**2, axis=1))
        return distances
    
    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

### K-Means on Sample Data

In [ ]:
# Generate sample data
np.random.seed(42)
X1 = np.random.randn(100, 2) + np.array([2, 2])
X2 = np.random.randn(100, 2) + np.array([-2, -2])
X3 = np.random.randn(100, 2) + np.array([2, -2])
X = np.vstack([X1, X2, X3])

# Apply K-Means from scratch
kmeans = KMeansFromScratch(n_clusters=3, random_state=42)
kmeans.fit(X)
labels = kmeans.labels

# Apply scikit-learn K-Means
kmeans_sklearn = KMeans(n_clusters=3, random_state=42)
labels_sklearn = kmeans_sklearn.fit_predict(X)

print("=== K-Means Clustering ===")
print(f"Centroids (from scratch):\n{kmeans.centroids}")
print(f"\nCluster distribution: {np.bincount(labels)}")

### Visualize K-Means Results

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7)
plt.scatter(kmeans.centroids[:, 0], kmeans.centroids[:, 1], c='red', marker='X', s=200)
plt.title('K-Means (From Scratch)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.subplot(1, 2, 2)
plt.scatter(X[:, 0], X[:, 1], c=labels_sklearn, cmap='viridis', alpha=0.7)
plt.scatter(kmeans_sklearn.cluster_centers_[:, 0], kmeans_sklearn.cluster_centers_[:, 1], c='red', marker='X', s=200)
plt.title('K-Means (Scikit-Learn)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')

plt.tight_layout()
plt.show()

## 3. Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Hierarchical Clustering using Scikit-Learn
hc = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_hc = hc.fit_predict(X)

plt.figure(figsize=(8, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels_hc, cmap='viridis', alpha=0.7)
plt.title('Hierarchical Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

# Dendrogram
plt.figure(figsize=(10, 6))
Z = linkage(X, method='ward')
dendrogram(Z, truncate_mode='lastp', p=30)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.show()

## 4. Dimensionality Reduction: PCA

In [ ]:
# Generate higher-dimensional data
np.random.seed(42)
X_high = np.random.randn(200, 5)

# Apply PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_high)

print("=== PCA Dimensionality Reduction ===")
print(f"Original shape: {X_high.shape}")
print(f"Reduced shape: {X_pca.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.4f}")

## 5. t-SNE Visualization

In [ ]:
from sklearn.datasets import load_iris

# Use iris dataset
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X_iris)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_iris[:, 0], X_iris[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('Original Iris Data')
plt.xlabel('Sepal Length')
plt.ylabel('Sepal Width')

plt.subplot(1, 2, 2)
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('t-SNE Visualization')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')

plt.tight_layout()
plt.show()

## 6. Market Basket Analysis

### Association Rule Mining with Apriori Algorithm

In [ ]:
# Create sample market basket data
transactions = [
    ['milk', 'bread', 'eggs'],
    ['bread', 'butter', 'eggs'],
    ['milk', 'bread', 'butter'],
    ['bread', 'eggs'],
    ['milk', 'eggs'],
    ['bread', 'milk', 'eggs'],
    ['butter', 'eggs'],
    ['milk', 'bread', 'butter', 'eggs'],
    ['milk', 'butter'],
    ['eggs', 'bread']
]

print("Sample Transactions:")
for i, t in enumerate(transactions[:5]):
    print(f"Transaction {i+1}: {t}")

### Encode Transactions

In [ ]:
# Encode transactions using TransactionEncoder
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df = pd.DataFrame(te_array, columns=te.columns_)

print("Encoded Transaction Data:")
print(df.head())

### Apply Apriori Algorithm

In [ ]:
# Find frequent itemsets with minimum support of 0.3
frequent_itemsets = apriori(df, min_support=0.3, use_colnames=True)

print("=== Frequent Itemsets ===")
print(frequent_itemsets.sort_values('support', ascending=False))

### Generate Association Rules

In [ ]:
# Generate association rules with minimum confidence of 0.5
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

print("\n=== Association Rules ===")
if len(rules) > 0:
    rules_display = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].sort_values('lift', ascending=False)
    rules_display['antecedents'] = rules_display['antecedents'].apply(lambda x: ', '.join(list(x)))
    rules_display['consequents'] = rules_display['consequents'].apply(lambda x: ', '.join(list(x)))
    print(rules_display.to_string(index=False))
else:
    print("No rules found with the given thresholds.")

### Larger Market Basket Dataset

In [ ]:
# Generate larger synthetic dataset
np.random.seed(42)
n_transactions = 100

items = ['milk', 'bread', 'eggs', 'butter', 'cheese', 'yogurt', 'rice', 'pasta', 'tomatoes', 'onions']

def generate_transaction():
    n_items = np.random.randint(2, 7)
    return list(np.random.choice(items, n_items, replace=False))

large_transactions = [generate_transaction() for _ in range(n_transactions)]

# Encode
te_large = TransactionEncoder()
te_array_large = te_large.fit_transform(large_transactions)
df_large = pd.DataFrame(te_array_large, columns=te_large.columns_)

print(f"Dataset: {n_transactions} transactions, {len(items)} items")
print(f"\nSample transactions:")
for i in range(3):
    print(f"{i+1}. {large_transactions[i]}")

### Frequent Itemsets (Larger Dataset)

In [ ]:
# Find frequent itemsets
frequent_large = apriori(df_large, min_support=0.1, use_colnames=True)
frequent_large = frequent_large.sort_values('support', ascending=False)

print("=== Frequent Itemsets (Support >= 0.1) ===")
frequent_large['itemsets'] = frequent_large['itemsets'].apply(lambda x: ', '.join(list(x)))
print(frequent_large.head(10).to_string(index=False))

### Association Rules (Larger Dataset)

In [ ]:
# Generate rules
rules_large = association_rules(frequent_large, metric="confidence", min_threshold=0.3)
rules_large = rules_large.sort_values('lift', ascending=False)

print("\n=== Top Association Rules ===")
top_rules = rules_large[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)
top_rules['antecedents'] = top_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
top_rules['consequents'] = top_rules['consequents'].apply(lambda x: ', '.join(list(x)))
print(top_rules.to_string(index=False))

### Visualize Rules

In [ ]:
if len(rules_large) > 0:
    plt.figure(figsize=(10, 6))
    plt.scatter(rules_large['support'], rules_large['confidence'], 
                alpha=0.6, s=rules_large['lift']*100, c='steelblue')
    plt.xlabel('Support')
    plt.ylabel('Confidence')
    plt.title('Association Rules: Support vs Confidence')
    plt.grid(True, alpha=0.3)
    plt.show()

## 7. Summary

### Key Concepts Covered:

1. **K-Means Clustering**: Unsupervised algorithm for partitioning data into K clusters
2. **Hierarchical Clustering**: Creates hierarchy of clusters with dendrogram visualization
3. **PCA**: Principal Component Analysis for dimensionality reduction
4. **t-SNE**: Non-linear dimensionality reduction for visualization
5. **Association Rule Mining**: Apriori algorithm for market basket analysis

### Key Metrics:
- **Support**: Frequency of itemset in transactions
- **Confidence**: P(B|A) - probability of B given A
- **Lift**: How much A increases the probability of B